# 第十二周整改与统计主线复现

本Notebook调用项目内可独立运行的脚本，在整改结果目录复现冻结counts后的统计主线。默认核对输入及结果校验和后使用已完成模型；首次实际拟合日志与每个模型的日志单独保留。需要强制重拟合时，在analysis及source控制命令中增加`--force-refit`。本轮不重跑01—06、改变主队列或进行嵌套表示训练。

本文件的自动执行不代替作者本人操作与10分钟口头汇报。


In [1]:
import os, subprocess
from pathlib import Path
import pandas as pd
root = Path.cwd().parent
assert (root / "data_processed/final_data.h5ad").is_file(), "从notebooks目录运行"
runtime = "C:/Users/asd123cheese/AppData/Local/Programs/Python/Python314/python.exe"
cache = root / "tmp/remediation_runtime"
for p in [cache, cache/"numba", cache/"mpl"]: p.mkdir(parents=True, exist_ok=True)
os.environ.update(TEMP=str(cache), TMP=str(cache), NUMBA_CACHE_DIR=str(cache/"numba"), MPLCONFIGDIR=str(cache/"mpl"), MPLBACKEND="Agg", OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
def run(script, *args):
    result = subprocess.run([runtime, "scripts/"+script, *args], cwd=root, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout[-6000:])
    if result.returncode: print(result.stderr[-6000:])
    assert result.returncode == 0, (script, result.returncode)


## 重建全部375列与重新核对DE


In [2]:
run('12_remediate_analysis.py', 'all')

Audit standardized files and rebuild all pseudobulk counts
standardized audit: Fat_checked_standardized.h5ad
standardized audit: Heart_and_Aorta_checked_standardized.h5ad
standardized audit: Kidney_checked_standardized.h5ad
standardized audit: Liver_checked_standardized.h5ad
standardized audit: Lung_checked_standardized.h5ad
standardized audit: spleen_marrow_checked_standardized.h5ad
standardized audit: TMS_checked_standardized.h5ad
Full raw-count reconstruction complete



## Spleen与Marrow来源重建及控制


In [3]:
run('12_remediate_source_controls.py')

Source control {'scenario': 'source_spleen', 'tissue': 'spleen/marrow', 'mct': 'lymphoid', 'n_units': 13, 'n_young': 3, 'n_old': 4, 'n_middle': 6, 'n_cells': 28468, 'status': 'complete', 'continuous': False, 'formula': '~sex+age_group', 'design_columns': 4, 'design_rank': 4, 'residual_df': 9, 'coefficient_unit': 'old vs young', 'n_genes': 9165, 'n_P_available': 9160, 'n_FDR_available': 9160, 'n_sig': 775, 'seconds': 22.21, 'result_sha256': '77fa01fda401a190e58f8bddc1537a4ec1d64c5e3a886590ad4ff01e0e37676b', 'input_fingerprint': '45785426f19d68369f1b89147cb3708685cd9fde9dc6744a5709d2baeae72ab1', 'fingerprint_recorded_after_fit_from_reconstructed_input': True}
Source control {'scenario': 'source_spleen', 'tissue': 'spleen/marrow', 'mct': 'myeloid', 'n_units': 13, 'n_young': 3, 'n_old': 4, 'n_middle': 6, 'n_cells': 3988, 'status': 'complete', 'continuous': False, 'formula': '~sex+age_group', 'design_columns': 4, 'design_rank': 4, 'residual_df': 9, 'coefficient_unit': 'old vs young', 'n_gen

## 全库富集及所有控制


In [4]:
run('12_remediate_enrichment.py', 'all')

Enrichment scope complete: all



## 汇总全部效应与失败记录


In [5]:
run('12_remediate_summary.py')

{
  "DE_scenarios": {
    "continuous_age": 7,
    "exclude_uncertain": 7,
    "male_only": 6,
    "min30": 5,
    "min50": 5,
    "primary_refit": 7,
    "source_marrow": 2,
    "source_spleen": 2,
    "subtype_macrophage": 3,
    "subtype_t_cell": 3
  },
  "n_fits": 47,
  "n_not_inferential": 13,
  "primary_DE_records": 1400,
  "primary_GO_tests": 26978,
  "enrichment_node_library_runs": 68,
  "Hallmark_significant_records": 117,
  "Reactome_significant_records": 559
}



In [6]:
checks = pd.read_csv(root/"results/remediation_20261103/reproduction_checks.tsv", sep="\t")
assert checks.passed.all()
de = pd.read_csv(root/"results/remediation_20261103/primary_DE_reproduction.tsv", sep="\t")
gsea = pd.read_csv(root/"results/remediation_20261103/primary_GSEA_reproduction.tsv", sep="\t")
assert len(de)==7 and len(gsea)==7
assert de.max_log2FC_error.eq(0).all() and de.max_P_error.eq(0).all() and de.max_FDR_error.eq(0).all()
assert gsea.max_NES_error.eq(0).all() and gsea.max_P_error.eq(0).all() and gsea.max_FDR_error.eq(0).all()
print("7-node DE and 26978-test GO GSEA reproduced exactly; author demonstration remains pending.")

7-node DE and 26978-test GO GSEA reproduced exactly; author demonstration remains pending.
